# <font color='#E8800A'>Import and load the data</font> <a class="anchor" id="import"></a>

In [41]:
import joblib
import numpy as np
import pandas as pd

from scipy.stats import chi2_contingency, spearmanr
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFECV
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)  # For reproducibility

# Saved by the pre-processing notebook (rows are sorted by date)
train = pd.read_csv("../data/pre-processed/train_clean.csv", index_col="ID")
target = "repeat_purchase_90d"
X, y = train.drop(columns=[target]), train[target]

preprocess = joblib.load("../data/pre-processed/preprocess.joblib")
time_split = TimeSeriesSplit(n_splits=5)

# purchase_date only gives the order of the rows; it is not a feature
features = [column for column in X.columns if column != "purchase_date"]
numeric_features = X[features].select_dtypes(include="number").columns.tolist()
categorical_features = [column for column in features if column not in numeric_features]

print(f"{len(X):,} rows, {len(features)} candidate features"
      f" ({len(numeric_features)} numeric, {len(categorical_features)} text)")

6,469 rows, 38 candidate features (29 numeric, 9 text)


# <font color='#E8800A'>Strategy</font> <a class="anchor" id="strategy"></a>

We start from the 38 candidate features left after pre-processing and decide in five steps:

1. **Identifier-like variables:** look critically at the variables that name a row or a customer, before anything else.
2. **Filter methods** (each feature on its own, no model):
   - *low variance:* the same value in more than 95% of rows → dropped
   - *relation with the target:* Spearman (numbers) or chi-square (text), p < 0.05 → **1 vote**
   - *redundant:* two features with Spearman correlation above 0.9 say the same thing → keep the one more related to the target, drop the other
3. **Wrapper method:** RFE with logistic regression → **1 vote**
4. **Embedded methods:** Lasso → **1 vote**; random forest importance higher than `random_noise` → **1 vote**
5. **Final decision:** keep a feature if it is not low-variance, not redundant, and gets **at least 2 of the 4 votes**. Then check the result with cross-validation.

No single method decides alone, because each has blind spots: filters ignore how features work together, RFE and Lasso split their choice between correlated features, and random forest importance favours continuous columns over 0/1 ones. So they vote.

Everything uses the training data only, never `test.csv`. Scores are F1 with `TimeSeriesSplit`, the same validation the models will use.

# <font color='#E8800A'>Identifier-like variables</font> <a class="anchor" id="identifiers"></a>

A variable that names a row or a customer can look predictive only because it lets the model recognise *who* it is. That doesn't help with new purchases, so each one is checked instead of being kept or dropped automatically.

In [42]:
raw = pd.read_csv("../data/raw/train.csv", index_col="ID").loc[X.index]

# Does the row ID or the customer number say anything about the target?
for name, values in {"ID": X.index.to_series(), "customer_id": raw["customer_id"]}.items():
    rho, p = spearmanr(values, y)
    print(f"{name:12s} Spearman with the target {rho:+.3f}  (p = {p:.1e})")

ID           Spearman with the target +0.012  (p = 3.5e-01)
customer_id  Spearman with the target -0.116  (p = 1.1e-20)


In [43]:
# The clientes.csv columns, for the customers in train: how many customers share each value?
clientes = pd.read_csv("../data/raw/clientes.csv", index_col="customer_id")
clientes = clientes.loc[clientes.index.isin(raw["customer_id"])].drop(columns="customer_type")

pd.DataFrame({
    "distinct values": clientes.nunique(),
    "customers per value": (len(clientes) / clientes.nunique()).round(1),
}).sort_values("distinct values", ascending=False)

,distinct values,customers per value
company_name,606,1.0
address,597,1.0
postal_code,516,1.2
street,264,2.3
city,259,2.3
postal_area,188,3.2
municipality,150,4.0
commercial_region,32,19.0
island,10,60.7
customer_since_year,10,60.7


To test `customer_id` with a model, we compare the cross-validated F1 with and without it. Two helpers are used for every comparison in this notebook:

In [44]:
def pipeline_for(keep, extra=None):
    """The pre-processing pipeline, using only the features in `keep` (plus an optional extra step)."""
    steps = [(name, step, [column for column in columns if column in keep])
             for name, step, columns in preprocess.transformers
             if any(column in keep for column in columns)]
    if extra:
        steps.append(extra)
    return clone(preprocess).set_params(transformers=steps)


models = {
    "logistic regression": LogisticRegression(max_iter=2000),
    "random forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                                            random_state=RANDOM_STATE, n_jobs=-1),
}


def cv_f1(data, pipeline):
    """Mean F1 over the 5 time-series folds, for each model."""
    return {
        name: cross_val_score(Pipeline([("preprocess", pipeline), ("model", model)]),
                              data, y, cv=time_split, scoring="f1").mean().round(3)
        for name, model in models.items()
    }

In [45]:
X_with_id = X.assign(customer_id=raw["customer_id"])
id_as_number = ("customer_id", StandardScaler(), ["customer_id"])
id_as_category = ("customer_id", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["customer_id"])

baseline = cv_f1(X, pipeline_for(features))
pd.DataFrame({
    "all features": baseline,
    "+ customer_id as a number": cv_f1(X_with_id, pipeline_for(features, id_as_number)),
    "+ customer_id as a category": cv_f1(X_with_id, pipeline_for(features, id_as_category)),
}).T

,logistic regression,random forest
all features,0.730,0.751
+ customer_id as a number,0.729,0.750
+ customer_id as a category,0.714,0.752


#### Decisions on the identifier-like variables

| variable | what it is | evidence | decision |
|---|---|---|---|
| `ID` | row number | no link with the target (Spearman +0.01, p = 0.35) | not a feature; kept as the index for the submission |
| `customer_id` | anonymized customer number | linked to the target (Spearman −0.12, p ≈ 10⁻²⁰), although the numbers mean nothing | **excluded**, see below |
| `company_name`, `address`, `postal_code` | from `clientes.csv` | about 1 customer per value: they *are* the customer | dropped in pre-processing |
| `street`, `city`, `postal_area`, `municipality` | from `clientes.csv` | 2 to 4 customers per value, and no link with the target once each customer counts once (exploration) | dropped in pre-processing |
| `commercial_region`, `island`, `customer_since_year`, `country_or_region` | from `clientes.csv` | 19 to 87 customers per value: real groups, not identifiers | kept as candidates, with a second look in the final decision |
| `purchase_date` | when the purchase happened | `test.csv` is entirely later than train, so a model can't learn from dates it has never seen | not a feature; only used to sort the rows for `TimeSeriesSplit` |
| `random_noise` | random numbers | no link with the target (exploration) | not a feature; used below as a benchmark for the random forest |

**Why `customer_id` is excluded, even though it is "linked" to the target:** an anonymized number can't carry information by itself. The link only means some frequent buyers happen to have certain numbers, so the model would be recognising *who* the customer is. The test confirms it adds nothing: as a number, F1 stays the same (0.729 vs 0.730 for logistic regression, 0.750 vs 0.751 for the random forest). As a category (one column per customer), it makes logistic regression worse (0.714) and doesn't help the forest (0.752). It also can't work for the 22 customers in `test.csv` that never appear in train. What we actually know about a customer is already in the history features (`prior_purchase_count`, `purchase_count_365d`, ...), and those also work for new customers.

# <font color='#E8800A'>Filter methods</font> <a class="anchor" id="filter"></a>

Each feature on its own, without a model.

## Low variance

A feature that is almost always the same value can't tell purchases apart. `VarianceThreshold` measures this with the variance, but our columns are on very different scales and some are text, so we use the same idea in a form that works for all of them: the share of rows with the most common value.

In [46]:
dominant_share = X[features].apply(
    lambda column: column.value_counts(normalize=True, dropna=False).iloc[0])

low_variance = dominant_share[dominant_share > 0.95].index.tolist()
print("low variance (dropped):", low_variance)
dominant_share.sort_values(ascending=False).head(8).round(3).to_frame("share of the most common value")

low variance (dropped): ['Total Desc_ Global', 'commercial_zone', 'warehouse', 'country_or_region']


,share of the most common value
Total Desc_ Global,0.983
commercial_zone,0.968
country_or_region,0.967
warehouse,0.967
invoice_count_current_purchase,0.945
Total Portes,0.837
payment_type,0.706
purchase_count_30d,0.671


## Relation with the target

Numbers: Spearman correlation (it works with skewed columns and doesn't assume a straight line). Text: chi-square test, as in the exploration notebook.

In [47]:
target_relation = pd.DataFrame({
    "Spearman": X[numeric_features].apply(lambda column: spearmanr(column, y, nan_policy="omit")[0]),
    "p": X[numeric_features].apply(lambda column: spearmanr(column, y, nan_policy="omit")[1]),
})
for column in categorical_features:
    table = pd.crosstab(X[column].fillna("Missing"), y)
    target_relation.loc[column, "p"] = chi2_contingency(table).pvalue

related = target_relation["p"] < 0.05
print(f"related to the target: {related.sum()} of {len(features)}")
print("not related:", related[~related].index.tolist())
target_relation.sort_values("p").round(4)

related to the target: 35 of 38
not related: ['Total Portes', 'Custo', 'document_series']


,Spearman,p
purchase_count_365d,0.4370,0.0000
purchase_count_180d,0.4061,0.0000
purchase_count_90d,0.3928,0.0000
spend_365d,0.3875,0.0000
prior_mean_gap_days,-0.4204,0.0000
spend_180d,0.3695,0.0000
spend_90d,0.3591,0.0000
prior_purchase_count,0.3456,0.0000
prior_total_spend,0.3316,0.0000
days_since_previous_purchase,-0.3153,0.0000


## Redundant features (correlation between features)

Some features are almost copies of each other, for example `Total a Pagar` is `Total Liquido` plus tax. Keeping both adds nothing and makes linear models unstable. Going from the feature most related to the target to the least, a feature is dropped if it is more than 0.9 correlated with one already kept.

In [48]:
correlation = X[numeric_features].corr(method="spearman").abs()
strongest_first = target_relation.loc[numeric_features, "Spearman"].abs().sort_values(ascending=False)

kept, redundant = [], {}
for feature in strongest_first.index:
    twin = next((other for other in kept if correlation.loc[feature, other] > 0.9), None)
    if twin is None:
        kept.append(feature)
    else:
        redundant[feature] = twin

pd.Series(redundant).to_frame("dropped, almost the same as")

,"dropped, almost the same as"
spend_90d,purchase_count_90d
prior_total_spend,prior_purchase_count
spend_30d,purchase_count_30d
prior_std_purchase_value,prior_max_purchase_value
Total a Pagar,Total Bruto
Total Liquido,Total Bruto


# <font color='#E8800A'>Wrapper methods</font> <a class="anchor" id="wrapper"></a>

RFE, Lasso and the random forest need numbers, so they work on the output of the pre-processing pipeline. A text feature becomes several 0/1 columns there, so a feature counts as chosen if **any** of its columns is chosen.

In [49]:
X_ready = clone(preprocess).fit_transform(X, y)


def original_feature(column):
    """'categorical__payment_type_Maybe Tomorrow Payment 9' -> 'payment_type'"""
    name = column.split("__", 1)[1].removeprefix("missingindicator_")
    if name in features:
        return name
    return next(feature for feature in categorical_features if name.startswith(feature + "_"))


feature_of = pd.Series([original_feature(column) for column in X_ready.columns], index=X_ready.columns)
print(f"{len(features)} features -> {X_ready.shape[1]} columns after the pipeline")

38 features -> 125 columns after the pipeline


## RFE (Recursive Feature Elimination)

Fits a logistic regression, removes the weakest column, and repeats. `RFECV` picks how many columns to keep by cross-validation (with our time-series folds).

In [50]:
rfe = RFECV(LogisticRegression(max_iter=2000), cv=time_split, scoring="f1").fit(X_ready, y)

rfe_vote = pd.Series(rfe.support_, index=X_ready.columns).groupby(feature_of).any()
print(f"RFE keeps {rfe.n_features_} of {X_ready.shape[1]} columns, from {rfe_vote.sum()} features")
print("not chosen:", rfe_vote[~rfe_vote].index.tolist())

RFE keeps 22 of 125 columns, from 11 features
not chosen: ['Custo', 'N_ Detalhes', 'Total Desc_ Global', 'Total Desc_ Linha', 'Total Imp_', 'Total Portes', 'Total a Pagar', 'country_or_region', 'customer_since_year', 'customer_tenure_days', 'days_since_previous_purchase', 'document_series', 'invoice_count_current_purchase', 'payment_type', 'prior_max_purchase_value', 'prior_mean_gap_days', 'prior_mean_purchase_value', 'prior_min_purchase_value', 'prior_purchase_count', 'prior_std_gap_days', 'prior_std_purchase_value', 'prior_total_spend', 'purchase_count_30d', 'spend_180d', 'spend_30d', 'spend_365d', 'spend_90d']


# <font color='#E8800A'>Embedded methods</font> <a class="anchor" id="embedded"></a>

The model picks the features itself while it trains.

## Lasso (L1 logistic regression)

The L1 penalty pushes the coefficients of unhelpful columns to exactly 0. The penalty strength is picked by cross-validation.

In [51]:
# l1_ratios=(1,) means a pure L1 (Lasso) penalty
lasso = LogisticRegressionCV(Cs=10, l1_ratios=(1,), solver="liblinear", cv=time_split, scoring="f1",
                             max_iter=2000, use_legacy_attributes=False).fit(X_ready, y)

lasso_vote = pd.Series(lasso.coef_[0] != 0, index=X_ready.columns).groupby(feature_of).any()
print(f"Lasso keeps {lasso_vote.sum()} of {len(features)} features (C = {lasso.C_:.3f})")
print("set to 0:", lasso_vote[~lasso_vote].index.tolist())

Lasso keeps 32 of 38 features (C = 0.359)
set to 0: ['Total Desc_ Global', 'Total Liquido', 'Total a Pagar', 'prior_total_spend', 'spend_180d', 'spend_90d']


## Tree-based importance

`random_noise` is pure noise, so it is added back here as a benchmark: a feature the forest finds **less** important than noise is not helping it. A text feature's importance is the sum over its 0/1 columns.

Caveat: this importance favours continuous columns (like the noise) over 0/1 columns, so text features are at a disadvantage. That's one reason it is only one vote of four.

In [52]:
X_with_noise = X_ready.assign(random_noise=raw["random_noise"].values)
forest = RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                                random_state=RANDOM_STATE, n_jobs=-1).fit(X_with_noise, y)

feature_of_with_noise = pd.concat([feature_of, pd.Series({"random_noise": "random_noise"})])
importance = (pd.Series(forest.feature_importances_, index=X_with_noise.columns)
              .groupby(feature_of_with_noise).sum().sort_values(ascending=False))

forest_vote = importance.drop("random_noise") > importance["random_noise"]
print(f"more important than random_noise: {forest_vote.sum()} of {len(features)} features")
importance.round(4).to_frame("importance")

more important than random_noise: 17 of 38 features


,importance
purchase_count_365d,0.0734
prior_mean_gap_days,0.0728
spend_365d,0.0582
purchase_count_180d,0.0493
spend_180d,0.0454
prior_std_gap_days,0.0426
purchase_count_90d,0.0370
prior_total_spend,0.0353
days_since_previous_purchase,0.0352
prior_purchase_count,0.0334


# <font color='#E8800A'>Final decision</font> <a class="anchor" id="decision"></a>

In [53]:
decision = pd.DataFrame({
    "related to target": related,
    "RFE": rfe_vote,
    "Lasso": lasso_vote,
    "forest > noise": forest_vote,
}).loc[features]
decision["votes"] = decision.sum(axis=1)
decision["low variance"] = decision.index.isin(low_variance)
decision["redundant"] = decision.index.isin(list(redundant))
decision["keep"] = (decision["votes"] >= 2) & ~decision["low variance"] & ~decision["redundant"]

voted = decision.index[decision["keep"]].tolist()
print(f"kept by the rule: {len(voted)} of {len(features)} features")
decision.sort_values(["keep", "votes"])

kept by the rule: 25 of 38 features


,related to target,RFE,Lasso,forest > noise,votes,low variance,redundant,keep
Total a Pagar,True,False,False,False,1,False,True,False
Total Portes,False,False,True,False,1,False,False,False
Total Desc_ Global,True,False,False,False,1,True,False,False
Custo,False,False,True,False,1,False,False,False
document_series,False,False,True,False,1,False,False,False
Total Liquido,True,True,False,False,2,False,True,False
prior_total_spend,True,False,False,True,2,False,True,False
spend_30d,True,False,True,False,2,False,True,False
spend_90d,True,False,False,True,2,False,True,False
country_or_region,True,False,True,False,2,True,False,False


## Second look at the customer-level features

`commercial_region`, `island` and `customer_since_year` come from `clientes.csv`: every purchase of a customer has the same value. In the exploration, their link with the target disappeared when each customer counted only once, so they may be standing in for "which customer is this". We check whether the models need them.

In [54]:
customer_level = [feature for feature in ["commercial_region", "island", "customer_since_year"] if feature in voted]
without_customer_level = [feature for feature in voted if feature not in customer_level]

pd.DataFrame({
    f"all features ({len(features)})": baseline,
    f"after the vote ({len(voted)})": cv_f1(X, pipeline_for(voted)),
    f"after the vote, without customer-level ({len(without_customer_level)})": cv_f1(X, pipeline_for(without_customer_level)),
}).T

,logistic regression,random forest
all features (38),0.730,0.751
after the vote (25),0.730,0.748
"after the vote, without customer-level (22)",0.733,0.750


## Final feature set

Dropping the three customer-level features doesn't lower the scores (0.733 vs 0.730, and 0.750 vs 0.748), which confirms the exploration: they were mostly standing in for the customer. So they are dropped too.

**22 features are kept**, out of 38:

| group | features |
|---|---|
| current purchase | `Total Bruto`, `Total Imp_`, `Total Desc_ Linha`, `N_ Detalhes`, `invoice_count_current_purchase` |
| recency and tenure | `days_since_previous_purchase`, `customer_tenure_days` |
| purchase history | `prior_purchase_count`, `prior_mean_purchase_value`, `prior_min_purchase_value`, `prior_max_purchase_value`, `prior_mean_gap_days`, `prior_std_gap_days` |
| recent activity | `purchase_count_30d`, `purchase_count_90d`, `purchase_count_180d`, `purchase_count_365d`, `spend_180d`, `spend_365d` |
| text | `payment_type`, `salesperson`, `transport_method` |

**16 are dropped**, each for a stated reason:

| reason | features |
|---|---|
| low variance (same value in more than 95% of rows) | `Total Desc_ Global`, `commercial_zone`, `warehouse`, `country_or_region` |
| redundant (almost a copy of a kept feature) | `Total a Pagar`, `Total Liquido` (≈ `Total Bruto`), `spend_30d`, `spend_90d`, `prior_total_spend` (≈ the matching purchase counts), `prior_std_purchase_value` (≈ `prior_max_purchase_value`) |
| only 1 vote of 4 | `Total Portes`, `Custo`, `document_series` |
| customer-level, not needed by the models | `commercial_region`, `island`, `customer_since_year` |

**Why this set:** it scores the same as all 38 features (F1 0.733 vs 0.730 for logistic regression, 0.750 vs 0.751 for the random forest) with 16 fewer features, and fewer encoded columns. A smaller model with the same score is easier to explain and less likely to overfit. Every feature left describes the purchase or the customer's behaviour, so it also works for customers the model has never seen.

The strongest features are about **how often the customer buys**: purchase counts over the last 90 to 365 days, and the typical gap between purchases. That makes sense for predicting whether they come back within 90 days.

*Caveat:* the methods above were run on all the training rows, so these cross-validated scores are slightly optimistic for the selected set. They are only used to compare sets, and the differences are small, so the conclusion holds.

The list and a pipeline that uses only these features are saved for the modelling notebook.

In [55]:
selected_features = without_customer_level

joblib.dump(selected_features, "../data/pre-processed/selected_features.joblib")
joblib.dump(pipeline_for(selected_features), "../data/pre-processed/preprocess_selected.joblib")
print(f"saved {len(selected_features)} features and the matching pipeline:")
print(selected_features)

saved 22 features and the matching pipeline:
['Total Bruto', 'Total Imp_', 'Total Desc_ Linha', 'N_ Detalhes', 'invoice_count_current_purchase', 'days_since_previous_purchase', 'customer_tenure_days', 'prior_purchase_count', 'prior_mean_purchase_value', 'prior_min_purchase_value', 'prior_max_purchase_value', 'prior_mean_gap_days', 'prior_std_gap_days', 'purchase_count_30d', 'purchase_count_90d', 'purchase_count_180d', 'spend_180d', 'purchase_count_365d', 'spend_365d', 'payment_type', 'salesperson', 'transport_method']
